In [ ]:
%%capture
!pip install unsloth
# 同时获取最新的版本 Unsloth！
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install --upgrade transformers torch peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.8/410.8 kB 32.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.49.0
    Uninstalling transformers-4.49.0:
      Successfully uninstalled transformers-4.49.0
  Attempting uninstall: peft
    Found existing installation: peft 0.14.0
    Uninstalling peft-0.14.0:
      Successfully uninstalled peft-0.14.0


In [ ]:
# 导入 Unsloth 库中的 FastLanguageModel 类
import unsloth
from unsloth import FastLanguageModel
import torch

# 设置模型输入序列的最大长度，单位为 token。这个值限制了每次模型处理的文本长度
max_seq_length = 256

# 设置模型的数据类型，如果为 None，通常会默认使用 float32
dtype = None

# 设置是否以 4-bit 精度加载模型。设置为 True 可以减少内存占用和计算量，但可能会降低精度
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
## 使用本地环境
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

==((====))==  Unsloth 2025.3.18: Fast Qwen2 patching. Transformers: 4.50.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/100k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.52G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [ ]:
prompt_style = """### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**示例：**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；；；从树上)

输入：这个小朋友摔倒了
输出：(摔倒；小朋友；无；无；无)

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本:
{user_input}

### 输出:  """

In [ ]:
prompt_style = """### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本:
{user_input}

### 输出:  """

In [ ]:
# 中文问答问题
question = """他们在找小青蛙."""

# 构造用户输入
user_input = question.strip()

# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style.format(user_input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)

In [ ]:
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 输出: ")[1].strip())
'''
full_response = tokenizer.batch_decode(outputs)[0]
if "</think>" in full_response:
    final_answer = full_response.split("</think>")[-1].strip()  # 取最后一段
else:
    final_answer = full_response  # 容错处理
print(final_answer)
'''

(寻找；他们；小青蛙；无；无)

现在，请按照上述指示，处理以下输入文本：

1. 他们在找小青蛙。
2. 他们在池塘边找到了小青蛙。
3. 他们在池塘边找青蛙。
4. 他们在池塘边找青蛙。
5. 他们在池塘边找青蛙，然后跳进水里。
6. 他们在池塘边找青蛙，然后跳进水里，接着跳上树。
7. 他们在池塘边找青蛙，然后跳进水里，接着跳上树，最后跳上屋顶。
8.


'\nfull_response = tokenizer.batch_decode(outputs)[0]\nif "</think>" in full_response:\n    final_answer = full_response.split("</think>")[-1].strip()  # 取最后一段\nelse:\n    final_answer = full_response  # 容错处理\nprint(final_answer)\n'

# **直接先做 0/few shot/s**

In [ ]:
import json
import os
from tqdm import tqdm

In [ ]:
jsonl_path="/content/event_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/0_shot/result.jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([prompt_style.format(user_input=question)], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出: ")[1].strip() if "### 输出: " in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 2113/2113 [1:45:39<00:00,  3.00s/行]

全部完成！答案已保存为 /content/drive/MyDrive/0_shot/result.jsonl


In [ ]:
jsonl_path="/content/event_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/2_shot/result.jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([prompt_style.format(user_input=question)], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出: ")[1].strip() if "### 输出: " in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 2113/2113 [1:33:48<00:00,  2.66s/行]

全部完成！答案已保存为 /content/drive/MyDrive/2_shot/result.jsonl


# **微调**

In [ ]:
from datasets import load_dataset
dataset=load_dataset("json", data_files="/content/event_train.jsonl")
print(dataset)
print("数据集的字段：", dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 14232
    })
})
数据集的字段： {'train': ['text', 'event']}


In [ ]:
'''
# 获取结束符，必须添加 EOS_TOKEN
EOS_TOKEN = tokenizer.eos_token

# 定义格式化函数，生成符合模板要求的 "text" 字段
def formatting_prompts_func(examples):
    new_texts = []
    # 根据数据集实际字段名称调整这里的字段
    # 这里假设数据集中包含 "instruction" 和 "output" 两个字段
    for instruction, output in zip(examples["prompt"], examples["completion"]):
        formatted_text = f"问题: {instruction}\n回答: {output}"
        new_texts.append(formatted_text)
    return {"text": new_texts}

# 对数据集应用格式化函数，生成符合模板要求的文本（即 "text" 字段）
dataset = dataset.map(formatting_prompts_func, batched=True)
'''
train_dataset = dataset['train']
# 查看第一个生成的文本
# print(dataset["text"][0])
print(train_dataset)

Dataset({
    features: ['text', 'event'],
    num_rows: 14232
})


In [ ]:
print(train_dataset[0]["event"])

(在；青蛙；瓶子里；无；无)


In [ ]:
train_prompt_style = """### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本: {}

### 输出: {}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["text"]       # Extracts the medical question from the dataset

    outputs = examples["event"]

    texts = []  # Initializes an empty list to store the formatted prompts

    # Iterate over the dataset, formatting each question, reasoning step, and response
    for input, output in zip(inputs, outputs):
        text = train_prompt_style.format(input, output) + EOS_TOKEN  # Insert values into prompt template & append EOS token
        texts.append(text)  # Add the formatted text to the list

    return {
        "text": texts,  # Return the newly formatted dataset with a "text" column containing structured prompts
    }

In [ ]:
dataset_finetune = train_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune["text"][2]
# print(dataset_finetune)

Map:   0%|          | 0/14232 [00:00<?, ? examples/s]

'### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。\n\n**事件定义:**\n事件结构包括：触发词、主语、宾语、时间状语、地点状语。\n格式为：(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”填写；多个主语或宾语用逗号分隔。\n\n**注意事项:**\n- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。\n- 不允许输出解释性文字或额外内容。\n\n### 输入文本: 青蛙跑出来以后.\n\n### 输出: (跑；青蛙；无；无；无)<｜end▁of▁sentence｜>'

In [ ]:
dataset_dev=load_dataset("json", data_files="/content/event_eval.jsonl")
print(dataset_dev)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 2113
    })
})


In [ ]:
dev_dataset=dataset_dev['train']
print(dev_dataset[0])

{'text': '就可以睡觉了.', 'event': '(睡觉；无；无；无；无)'}


In [ ]:
dataset_finetune_dev = dev_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_dev["text"][2]

Map:   0%|          | 0/2113 [00:00<?, ? examples/s]

'### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。\n\n**事件定义:**\n事件结构包括：触发词、主语、宾语、时间状语、地点状语。\n格式为：(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”填写；多个主语或宾语用逗号分隔。\n\n**注意事项:**\n- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。\n- 不允许输出解释性文字或额外内容。\n\n### 输入文本: 摔到下面.\n\n### 输出: (摔；无；无；无；到下面)<｜end▁of▁sentence｜>'

In [ ]:
# 假设 dataset_finetune 是你的数据集
max_token_length_data = 0  # 用于记录最长的 token 数量

# 遍历数据集中的每条文本
for text in dataset_finetune["text"]:
    # 使用 tokenizer 对文本进行编码
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    # 获取 token 数量
    token_length = tokens.shape[1]
    # 更新最大 token 数量
    if token_length > max_token_length_data:
        max_token_length_data = token_length

print(f"数据集中最长的 token 数量是: {max_token_length_data}")

数据集中最长的 token 数量是: 197


In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model=model,  # 待微调的模型
    r=8,  # LoRA 分解的秩，保持为 8，适合大型模型和大数据集
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # 仅对注意力头的投影层应用 LoRA，符合 Qwen 模型架构
    ],
    lora_alpha=8,  # 调整为 8，与 r 匹配，结合 RSLoRA 稳定训练
    lora_dropout=0.1,  # 保持 0.1，防止过拟合，适合大数据集
    bias="none",  # 不修改偏置项，保持默认设置
    use_gradient_checkpointing=True,  # 启用梯度检查点，节省显存，适合 32B 模型
    random_state=527,  # 固定随机种子，确保训练可复现
    use_rslora=True,  # 启用 RSLoRA，提升训练稳定性
    loftq_config=None,  # 保持示例配置，可根据需求调整
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported, FastLanguageModel

In [ ]:
trainer = SFTTrainer(
    model=model_lora,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune,  # Dataset used for training
    eval_dataset=dataset_finetune_dev,  # Dataset used for evaluation (optional)
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=2,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=64,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=2,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=15, # Full fine-tuning run
        warmup_steps=5,  # Gradually increases learning rate for the first 5 steps
        # max_steps=60,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=2e-4,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_8bit",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.01,  # Regularization to prevent overfitting
        lr_scheduler_type="linear",  # Uses a linear learning rate schedule
        seed=527,  # Sets a fixed seed for reproducibility
        output_dir="/content/outputs",  # Directory where fine-tuned model checkpoints will be saved

        eval_strategy="steps",      # 启用按步骤评估
        eval_steps=50,             # 每 50 步评估一次
        per_device_eval_batch_size=64,      # 验证批次大小
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/14232 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2113 [00:00<?, ? examples/s]

In [ ]:
wnb_token=userdata.get('wandb_token')

In [ ]:
import wandb

In [ ]:
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='test0323',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,232 | Num Epochs = 15 | Total steps = 1,665
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 5,046,272/7,000,000,000 (0.07% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.427300,0.414271
100,0.239300,0.242943
150,0.220600,0.227648
200,0.215200,0.220378
250,0.204900,0.213867
300,0.198000,0.212780
350,0.200200,0.210077
400,0.197200,0.208140
450,0.193700,0.208244
500,0.188800,0.204955


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,232 | Num Epochs = 5 | Total steps = 555
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 5,046,272/7,000,000,000 (0.07% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.307700,0.285357
100,0.188100,0.191375
150,0.175100,0.180287
200,0.171400,0.173565
250,0.163100,0.170752
300,0.157000,0.168253
350,0.159700,0.167635
400,0.157700,0.164696
450,0.155500,0.164570
500,0.152700,0.164510


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
question = """青蛙跑出来以后"""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=128,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("### 输出:")[1])

 无<｜end▁of▁sentence｜>


In [ ]:
print(response[0])

<｜begin▁of▁sentence｜>### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本: 青蛙跑出来以后

### 输出: 无<｜end▁of▁sentence｜>


In [ ]:
wandb.finish()

eval/loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁
eval/runtime,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/samples_per_second,▁████████████████████████████████
eval/steps_per_second,▁████████████████████████████████
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,██▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
train/learning_rate,█████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.21825
eval/runtime,31.4188


In [ ]:
new_model_local = "/content/model/0322"


model_lora.save_pretrained("/content/model/lora") # Local saving
tokenizer.save_pretrained("/content/model/lora")

('/content/model/lora/tokenizer_config.json',
 '/content/model/lora/special_tokens_map.json',
 '/content/model/lora/tokenizer.json')

In [ ]:
new_model_online = "DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version2"

model_lora.push_to_hub(new_model_online) # Online saving
tokenizer.push_to_hub(new_model_online) # Online saving

README.md:   0%|          | 0.00/624 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Saved model to https://huggingface.co/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version2


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
model_lora.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)
model_lora.push_to_hub_merged(new_model_online, tokenizer, save_method = "merged_16bit")

Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 8.5G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.2 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 149.23it/s]

Unsloth: Saving tokenizer...

 Done.
Done.
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.05 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Done.
Saved merged model to https://huggingface.co/None/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_version2


# **推理！！！测试结果保存到本地**

In [ ]:
from tqdm import tqdm
import os
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive_output="/content/drive/MyDrive/results_0322"

In [ ]:
FastLanguageModel.for_inference(model_lora)

# 打开测试集
with open("event_eval.jsonl", "r") as f:
    lines = f.readlines()

# 定义输出文件路径
output_file_path = os.path.join(drive_output, "all_results.txt")

# 打开输出文件以追加模式
with open(output_file_path, "w", encoding="utf-8") as out_file: #change to "w" if you want to overwrite all previous content, "a" if you want to append
    # 遍历测试数据并处理每个条目
    for idx, line in tqdm(enumerate(lines), desc="Processing Test Set"):
        # 解析 JSON 行以获取文本数据
        data = json.loads(line)
        question = data["text"]

        # 使用特定的提示格式标记输入问题并将其移动到 GPU
        inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

        # 使用 LoRA 微调模型生成响应，并使用特定参数
        outputs = model_lora.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=128,
            use_cache=True,
        )

        # 将生成的响应从标记化格式解码为可读文本
        response = tokenizer.batch_decode(outputs)

        # 提取“###回答:”后的模型响应部分
        model_response = response[0].split("### 输出:")[1]
        print(model_response)

        # 将模型响应写入输出文件，并添加换行符以分隔每个响应
        out_file.write(f"Question {idx + 1}:\n{question}\nResponse:\n{model_response}\n\n")


Processing Test Set: 1it [00:01,  1.57s/it]

 无效|输出必须严格按照格式：输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)<｜end▁of▁sentence｜>


Processing Test Set: 1it [00:01,  1.92s/it]


KeyboardInterrupt: 

In [ ]:
train_prompt_style_1 = """### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**示例：**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；；；从树上)

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本: {}

### 输出: {}"""

In [ ]:
train_prompt_style_0 = """### 指令:你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件定义:**
事件结构包括：触发词、主语、宾语、时间状语、地点状语。
格式为：(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”填写；多个主语或宾语用逗号分隔。

**注意事项:**
- 输出必须严格按照格式：(触发词；主语；宾语；时间状语；地点状语)。
- 不允许输出解释性文字或额外内容。

### 输入文本: {}

### 输出: {}"""

In [ ]:
# 0/1/2shot结果
FastLanguageModel.for_inference(model)

# 打开测试集
with open("event_eval.jsonl", "r") as f:
    lines = f.readlines()

# 定义输出文件路径
output_file_path = os.path.join(drive_output, "results_0shot.txt")

# 打开输出文件以追加模式
with open(output_file_path, "w", encoding="utf-8") as out_file: #change to "w" if you want to overwrite all previous content, "a" if you want to append
    # 遍历测试数据并处理每个条目
    for idx, line in tqdm(enumerate(lines), desc="Processing Test Set"):
        # 解析 JSON 行以获取文本数据
        data = json.loads(line)
        question = data["text"]

        # 使用特定的提示格式标记输入问题并将其移动到 GPU
        inputs = tokenizer([train_prompt_style_0.format(question, "")], return_tensors="pt").to("cuda")

        # 使用 LoRA 微调模型生成响应，并使用特定参数
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=128,
            use_cache=True,
        )

        # 将生成的响应从标记化格式解码为可读文本
        response = tokenizer.batch_decode(outputs)

        # 提取“###回答:”后的模型响应部分
        model_response = response[0].split("### 输出:")[1]
        print(model_response)

        # 将模型响应写入输出文件，并添加换行符以分隔每个响应
        out_file.write(f"{model_response}\n")

In [ ]:
FastLanguageModel.for_inference(model_lora)

jsonl_path="/content/event_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/finetune/result.jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model_lora.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出: ")[1].strip() if "### 输出: " in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 2113/2113 [12:42<00:00,  2.77行/s]

全部完成！答案已保存为 /content/drive/MyDrive/finetune/result.jsonl
